 SANAM ASGHARI

# 1. Data Extract with BeautifulSoup and Selenium

## 1.1. Launch

In [36]:

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup
import time

import pandas as pd 



In [88]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, 10)

driver.get("https://jobinja.ir/jobs/-%D8%A8%D8%B1%D9%86%D8%A7%D9%85%D9%87-%D9%86%D9%88%DB%8C%D8%B3-%D9%BE%D8%A7%DB%8C%D8%AA%D9%88%D9%86-python-developer")


## 1.2. Pagination

In [89]:
pages_html = []

for page in range(3):

    wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".o-listView__item")
        )
    )
    
    pages_html.append(driver.page_source)

    # Next page
    if page < 2:
        next_btn = wait.until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'a[rel="next"]')
            )
        )

        driver.execute_script("arguments[0].click();", next_btn)

        wait.until(EC.staleness_of(next_btn))

driver.quit()

## 1.3. Extract

In [90]:
all_jobs = []

for html in pages_html:
    # Receive HTML 
    soup = BeautifulSoup(html, "html.parser")
    
    #select jobs items 
    jobs = soup.select(".o-listView__item")

    for job in jobs:

        # Title
        title_tag = job.select_one("a.c-jobListView__titleLink")
        title = title_tag.get_text(strip=True) if title_tag else ""

        # Meta information
        meta = job.select("li.c-jobListView__metaItem")

        company = meta[0].get_text(strip=True) if len(meta) > 0 else ""
        city = meta[1].get_text(strip=True) if len(meta) > 1 else ""

        all_jobs.append({
            "title": title,
            "company": company,
            "city": city
        })

In [91]:
all_jobs

[{'title': '', 'company': '', 'city': ''},
 {'title': 'برنامه\u200cنویس پایتون (Python/Django-اصفهان)',
  'company': 'دانش بنیان مهندسی ارتباطی پیام پرداز | Payam Pardaz',
  'city': 'اصفهان، اصفهان'},
 {'title': 'توسعه دهنده پایتون (Python Software Engineer-AI Team)',
  'company': 'رهند هوشمند نوین داده | Rahand Hoshmand Novin Dadeh',
  'city': 'تهران، تهران'},
 {'title': 'برنامه\u200cنویس Python (اصفهان)',
  'company': 'باسا | BASA',
  'city': 'اصفهان، اصفهان'},
 {'title': 'Full-Stack Python Developer',
  'company': 'سدرا پرو | Sedrapro',
  'city': 'تهران، تهران'},
 {'title': 'برنامه\u200cنویس (Python (Django',
  'company': 'تحلیل گران شبکه آریانا | Ariana',
  'city': 'تهران، تهران'},
 {'title': 'برنامه\u200cنویس Back-End) Python)',
  'company': 'پارس پویش فن آور | Pars Pooyesh Fanavar',
  'city': 'تهران، تهران'},
 {'title': '(Senior Python Developer (Machine Learning',
  'company': 'سدرا پرو | Sedrapro',
  'city': 'تهران، تهران'},
 {'title': '(Senior Back-End Developer (Python',
  'c

## 1.4. Output

In [92]:
job_df = pd.DataFrame.from_dict(all_jobs)
job_df

,title,company,city
0,,,
1,برنامه‌نویس پایتون (Python/Django-اصفهان),دانش بنیان مهندسی ارتباطی پیام پرداز | Payam P...,اصفهان، اصفهان
2,توسعه دهنده پایتون (Python Software Engineer-A...,رهند هوشمند نوین داده | Rahand Hoshmand Novin ...,تهران، تهران
3,برنامه‌نویس Python (اصفهان),باسا | BASA,اصفهان، اصفهان
4,Full-Stack Python Developer,سدرا پرو | Sedrapro,تهران، تهران
...,...,...,...
58,برنامه‌نویس Front-End (خرم آباد),آپاسای داده سیستم | Apasai Dade System,لرستان، خرم آباد
59,متخصص پردازش زبان طبیعی (مشهد),گروه نرم افزاری پارت | Part Software Group,خراسان رضوی، مشهد
60,Team Lead) Senior Full Stack Software Engineer...,رادشید | Radshid,اصفهان، اصفهان
61,برنامه‌نویس ارشد Golang (دورکاری),ایزورا | eSora,تهران، تهران


In [93]:
job_df.isnull().sum()

title      0
company    0
city       0
dtype: int64

In [94]:
job_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    63 non-null     str  
 1   company  63 non-null     str  
 2   city     63 non-null     str  
dtypes: str(3)
memory usage: 1.6 KB


In [96]:
output = job_df.to_csv("job.csv")